# Import Packages

I'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `pandas` A powerful Python library for data manipulation and analysis.
- `seaborn` A data visualization library based on matplotlib.
- `scikit-learn` A comprehensive library for machine learning in Python.
- `kaggle` Using Kaggle API to download data.

In [758]:
import numpy as np
import pandas as pd
import kagglehub
import os
import re
import pickle
from collections import defaultdict, Counter


# Download Data

In [759]:
# Download latest version
path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")

print(os.listdir(path))

['used_cars.csv']


In [760]:
# Load the dataset
df = pd.read_csv(os.path.join(path, "used_cars.csv"))

In [761]:
## Path for assets
assets_dir = os.path.join('.', 'assets')

os.makedirs(assets_dir, exist_ok=True)

In [762]:
with open(os.path.join(assets_dir,'fuel_type_map.pkl'), 'rb') as f:
        fuel_type_map_1 = pickle.load(f)

# Data preprocessing

## Check data

In [763]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [764]:
## check for missing values
df.isna().sum()

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64

## 📝 Initial Data Glance – Observations & Next Steps

### Overview

- **Columns:**  
  The dataset contains **12 columns**: `brand`, `model`, `model_year`, `milage`, `fuel_type`, `engine`, `transmission`, `ext_col`, `int_col`, `accident`, `clean_title`, `price`.

- **Data Types:**  
  Most columns are **object type** (categorical or text). Data transformation will be required (label encoding, one-hot encoding, parsing text fields).

- **Brand & Model:**  
  - Some `model` values are duplicated or concatenated (e.g., `RX 350 RX 350`).
  - Will inspect and **clean/split/merge brand and model** to ensure unique, consistent values.

- **Feature Extraction:**  
  - Columns such as `engine` and `transmission` contain multiple details (e.g., horsepower, engine size, cylinder count, speed type) that can be **parsed into new features**.

- **Missing Values:**  
  - Nulls detected in several columns (e.g., `clean_title`).  
  - Will analyze missingness and apply appropriate imputation (mode, new category, or predictive imputation).

- **Target Variable:**  
  - The `price` column is a string with currency symbol and commas—needs to be cleaned and converted to numeric.

- **Formatting Issues:**  
  - Fields like `milage` and `price` contain units/symbols (e.g., "mi.", "$", ",")—will remove for numeric conversion.
  - `accident` and `clean_title` are categorical but may need binarization or mapping.

---

### Next Steps

- Clean and standardize all categorical and text fields.
- Parse and extract features from `engine` and `transmission` columns.
- Handle missing values with suitable imputation strategies.
- Convert `price` and `milage` to numeric types.
- Ensure brand/model consistency for analysis and modeling.

## Column Transformation

### Brand and Model

In [765]:
brand_list = sorted(df['brand'].unique())

by_letter = defaultdict(list)
for brand in brand_list:
    by_letter[brand[0].upper()].append(brand)
    

for letter in by_letter.keys():
    print(f"{letter}:{', '.join(by_letter[letter])}")

A:Acura, Alfa, Aston, Audi
B:BMW, Bentley, Bugatti, Buick
C:Cadillac, Chevrolet, Chrysler
D:Dodge
F:FIAT, Ferrari, Ford
G:GMC, Genesis
H:Honda, Hummer, Hyundai
I:INFINITI
J:Jaguar, Jeep
K:Karma, Kia
L:Lamborghini, Land, Lexus, Lincoln, Lotus, Lucid
M:MINI, Maserati, Maybach, Mazda, McLaren, Mercedes-Benz, Mercury, Mitsubishi
N:Nissan
P:Plymouth, Polestar, Pontiac, Porsche
R:RAM, Rivian, Rolls-Royce
S:Saab, Saturn, Scion, Subaru, Suzuki, smart
T:Tesla, Toyota
V:Volkswagen, Volvo


---
🏷️ Brand Name Inconsistencies

While reviewing the car brands, I noticed that several are missing their full names:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

Let’s take a closer look at these brands to ensure correct and consistent naming throughout the dataset.

In [766]:
df[df['brand'] == 'Alfa'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa,Romeo Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa,Romeo Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa,Romeo Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa,Romeo Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa,Romeo Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [767]:
df[df['brand'] == 'Aston'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston,Martin DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston,Martin DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston,Martin DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston,Martin V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [768]:
df[df['brand'] == 'Land'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land,Rover Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land,Rover LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land,Rover Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land,Rover LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land,Rover Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---
### 🔧 Brand Name Corrections Needed

As predicted, these three brands require their brand and model names to be fixed for consistency:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

In [769]:
### Fixing Brand Name Inconsistencies

df_update = df.copy()

## Replace inconsistent brand names with full names
df_update['brand'] = df_update['brand'].replace({
    'Alfa': 'Alfa Romeo',
    'Aston': 'Aston Martin',
    'Land': 'Land Rover'
})

## Remove first word from model names for these brands
brand_name = ['Alfa Romeo', 'Aston Martin', 'Land Rover']

for brand in brand_name:

    df_update.loc[df_update['brand'] == brand, 'model'] = df_update.loc[df_update['brand'] == brand, 'model'].str.split().apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) and len(x) > 1 else '')


In [770]:
df_update[df_update['brand'] == 'Alfa Romeo'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa Romeo,Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa Romeo,Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa Romeo,Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa Romeo,Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa Romeo,Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [771]:
df_update[df_update['brand'] == 'Aston Martin'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston Martin,DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston Martin,DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston Martin,DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston Martin,V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston Martin,V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [772]:
df_update[df_update['brand'] == 'Land Rover'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land Rover,Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land Rover,LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land Rover,Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land Rover,LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land Rover,Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---

Now that I’ve fixed the brand names, let’s address another issue: **duplicate model names**.  
For example, in row 2 I saw `Lexus RX 350 RX 350` as a model. These duplicates inflate the number of unique model groups.

**Next step:**  
Clean the `model` column to remove repeated names and reduce redundancy in our model grouping.

### 🔧 Clean Model names

In [773]:
# Remove duplicate model names
# For example, 'RX 350 RX 350' should be 'RX 350'
df_update['model'] = df_update['model'].apply(lambda x: ' '.join(dict.fromkeys(x.split())))

In [ ]:
## particularly for BMW models, clean up mutant model names with duplicated numeric badges
def clean_bmw_model(model):
    """
    Cleans up mutant BMW model names with duplicated numeric badges.
    Examples:
        '330 330i xDrive'         -> '330 i xDrive'
        'M550 M550i xDrive'       -> 'M550 i xDrive'
        '440 Gran Coupe 440i xDrive' -> '440 Gran Coupe i xDrive'
        '428 Gran Coupe 428i xDrive SULEV' -> '428 Gran Coupe i xDrive SULEV'
    Only works if the second badge starts with the first (e.g., '330' in '330i').
    """
    tokens = str(model).split()
    if len(tokens) < 2:
        return model
    for i in range(1, len(tokens)):
        if tokens[i].startswith(tokens[0]) and tokens[i].endswith('i'):
            suffix = tokens[i][len(tokens[0]):]  # Should just be 'i'
            tokens = tokens[:i] + [suffix] + tokens[i+1:]
            # Clean up empty tokens and extra spaces, just in case
            return ' '.join([t for t in tokens if t])
    return model

In [775]:
## Apply the cleaning function to BMW models
mask = df_update['brand'] == 'BMW'
df_update.loc[mask, 'model'] = df_update.loc[mask, 'model'].apply(clean_bmw_model)


### Extract numbers columns

In [776]:
col_names = ['milage', 'price']

def clean_numeric_col(series):
    """
    Clean numeric columns by removing non-numeric characters and converting to float.
    """
    return (
        series.astype(str)  # Ensure the series is of string type
        .str.replace(r'[^\d.]', '', regex=True)  # Remove non-numeric characters except digits and "."
        .astype(float)  # Convert to float
    )

# Apply the cleaning function to the specified columns
for col in col_names:
    df_update[col] = clean_numeric_col(df[col])

🔢 Now, both `milage` and `price` have been successfully converted to numerical columns.

---

### Individual columns
#### ⛽ Fuel Type: Data Cleaning Needed

In [777]:
df_update['fuel_type'].unique()

array(['E85 Flex Fuel', 'Gasoline', 'Hybrid', nan, 'Diesel',
       'Plug-In Hybrid', '–', 'not supported'], dtype=object)

The `fuel_type` column contains multiple categories, null values, and some unsupported or placeholder entries (e.g., `nan`, `'–'`, `'not supported'`).  
I’ll need to clean and unify these values for consistent analysis and modeling.

In [778]:
mask = df_update['fuel_type'].isin(['–', 'not supported']) | df_update['brand'].isna()
df.loc[mask, ['brand', 'model', 'fuel_type']]['brand'].value_counts()

brand
Dodge            8
Toyota           5
Ford             5
Mazda            4
Chrysler         3
Chevrolet        3
Cadillac         3
Nissan           3
Porsche          2
Acura            2
Mercedes-Benz    2
Rolls-Royce      1
Mercury          1
Volvo            1
Jaguar           1
Jeep             1
Honda            1
GMC              1
Name: count, dtype: int64

Tesla, Lucid, and Rivian are pure electric brands—missing `fuel_type` values for these can be safely filled as `'Electric'`.  
For all other brands, I’ll need to inspect the model before imputing the correct fuel type.

In [779]:
mask = (df_update['brand'].isin(['Tesla', 'Lucid', 'Rivian']) & 
        (df_update['fuel_type'].isin(['–', 'not supported']) |
         df_update['fuel_type'].isna()))

# Fill missing fuel_type for electric brands
df_update.loc[mask, 'fuel_type'] = 'Electric'

In [780]:
## Check if same models already have fuel_type
## Load it back
try:
    with open(os.path.join(assets_dir, 'fuel_type_map.pkl'), 'rb') as f:
        fuel_type_map = pickle.load(f)
except FileNotFoundError:
    print("File not found, creating new fuel_type_map.")
    
    mask = (~df_update['fuel_type'].isin(['–', 'not supported'])) & (~df_update['fuel_type'].isna())
    fuel_type_map = (
        df_update[mask]
        .groupby(['brand', 'model'])['fuel_type']
        .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
        .to_dict()
    )

In [781]:
## Impute fuel_type

def impute_fuel_type(row, fuel_type_map):
    """
    Impute fuel_type based on brand and model.

    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        str: The imputed transmission speeds.
    """
    mask = (row['fuel_type'] in ['–', 'not supported']) | (pd.isna(row['fuel_type']))
    if mask:
        key = (row['brand'], row['model'])
        return fuel_type_map.get(key, row['fuel_type'])
    else:
        return row['fuel_type']
    
## Apply the imputation function to the DataFrame
df_update['fuel_type'] = df_update.apply(lambda row: impute_fuel_type(row, fuel_type_map), axis=1)


In [ ]:
# # update fuel_type for other brands that's not available in the fuel_type_map

# # Mapping dictionary: brand, model -> fuel type
# map_fuel_type = {
    
# }

# ## Update fuel_type dictionary 
# fuel_type_map.update(map_fuel_type)

# # Save the new transmission type mapping
# with open(os.path.join(assets_dir,'fuel_type_map.pkl'), 'wb') as f:
#     pickle.dump(fuel_type_map, f)

In [783]:
# def assign_fuel_type(row, fuel_type_map):
#     if pd.isna(row['fuel_type']) or row['fuel_type'] in ['–', 'not supported']:
#         key = (row['brand'], row['model'])

#         return fuel_type_map.get(key, row['fuel_type'])
#     else:
#         return row['fuel_type']
    
# # Apply the mapping to the DataFrame
# df_update['fuel_type'] = df_update.apply(assign_fuel_type, axis=1)

✅ Fuel Type Cleanup Complete

All missing and inconsistent `fuel_type` values have been handled.  
The column is now clean and ready for analysis and modeling.

---

#### ⚙️ Transmission Data: Standardization & Feature Extraction

The `transmission` column contains various types and naming conventions for similar transmissions.  
To ensure consistency and improve analysis, we’ll standardize these values and extract two separate features:
- **Transmission Type** (e.g., Automatic, Manual, CVT)
- **Number of Speeds** (e.g., 6, 8, Single-Speed)

In [ ]:
## Identify and Standardize Transmission Types
"""
This step will focus on standardizing the values with both speed and mannual or automatic transmission.
Two new columns will be created:
- **Transmission Type** (e.g., Automatic, Manual, CVT)
- **Number of Speeds** (e.g., 6, 8, Single-Speed
"""

def extract_transmission_type(val):
    """
    Extracts the transmission type from the given value.

    Args:
        val (str): The transmission value to extract from.

    Returns:
        str: The transmission type ('Automatic', 'Manual', 'CVT', or 'Other').
    """
    v = str(val).lower()
    if any(keyword in v for keyword in ['automatic', 'a/t', 'auto', 'dual-clutch',
                                         'steptronic', 'dct', 'pdk', 'at', 'dual shift mode', 
                                         'overdrive switch', 'single-speed']):
        return 'Automatic'
    if any(keyword in v for keyword in ['manual', 'm/t', 'mt']):
        return 'Manual'
    if any(keyword in v for keyword in ['cvt', 'variable']):
        return 'CVT'
    return 'Other'


def extract_transmission_speeds(val):
    """
    Extracts the number of speeds from the given transmission value.

    Args:
        val (str): The transmission value to extract from.

    Returns:
        int: The number of speeds (e.g., 6), or 1 for single-speed, or None if not applicable.
    """
    v = str(val).lower()
    match = re.search(r'(\d+)[-\s]?(?:speed|spd)', v)
    if match:
        return int(match.group(1))
    if 'single-speed' in v or 'single speed' in v:
        return 1
    numbers = re.search(r'(\d+)', v)
    if numbers:
        return int(numbers.group(1))
    return None

💡 **Why Keep “CVT” as Its Own Category?**

1. **Mechanically Different**  
   - **Traditional Automatic:** Uses a set of gears, shifts through them automatically.  
   - **CVT:** No gears—uses pulleys and belts for infinite gear ratios.

2. **User Experience is Different**  
   - CVTs drive differently. No gear shifts, “rubber band” feel.  
   - Impacts consumer reviews, performance, and pricing.

3. **Manufacturer & Industry Reporting**  
   - Specs, consumer guides, and data vendors *always* break out “CVT” separately from “Automatic”.  
   - Lumping them together can hide meaningful patterns (pricing, reliability, satisfaction, etc).

4. **Modeling & Analytics**  
   - Some buyers specifically want to avoid (or seek out) CVTs.  
   - Resale value, repair costs, and reliability trends can differ significantly.  
   - **Keeping “CVT” as its own class maintains transparency and analytic flexibility.**

In [785]:
## Apply the Extraction Functions
df_update['transmission_type'] = df_update['transmission'].apply(extract_transmission_type)
df_update['transmission_speeds'] = df_update['transmission'].apply(extract_transmission_speeds)

In [786]:
df_update['transmission_type'].value_counts() 

transmission_type
Automatic    3557
Manual        373
CVT            67
Other          12
Name: count, dtype: int64

In [787]:
df_update['transmission_speeds'].isna().sum()

1832

In [788]:
## check uncategorized transmission types
mask = df_update['transmission_type'] == 'Other'

df_update.loc[mask,['model_year', 'brand', 'model', 'transmission', 'transmission_type']]

,model_year,brand,model,transmission,transmission_type
5,2016,Acura,ILX 2.4L,F,Other
269,2022,Acura,TLX w/A-Spec Package,2,Other
476,2023,Acura,MDX w/Technology Package,F,Other
516,2022,Acura,MDX w/Technology Package,2,Other
536,2017,Porsche,911 Carrera S,–,Other
855,1974,Ford,Bronco,–,Other
916,2018,Porsche,911 Carrera 4S,–,Other
1236,2019,Toyota,Tacoma TRD Pro,6-Speed,Other
1356,2021,Lamborghini,Aventador SVJ Base,7-Speed,Other
1615,2023,Rolls-Royce,Phantom,–,Other


As expected, some vehicles have incomplete transmission information.
- **1832 rows** are missing transmission speeds.
- **12 rows** are missing transmission type.

These missing values will need to be reviewed and corrected for accurate analysis.

##### Fix transmission type

In [789]:
## Check if same models already have transmission types
try:
    with open(os.path.join(assets_dir, 'transmission_type_map.pkl'), 'rb') as f:
        transmission_type_map = pickle.load(f)
except FileNotFoundError:
    transmission_type_map = (
        df_update[df_update['transmission_type'] != 'Other']
        .groupby(['model_year', 'brand', 'model'])['transmission_type']
        .agg(lambda x: x.mode()[0] if not x.mode().empty else 'Other')
        .to_dict()
    )

In [790]:
## Impute transmission_type
def impute_transmission_type(row, transmission_type_map):
    """
    Impute the transmission type based on the brand and model.
    
    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        str: The imputed transmission type.
    """
    if row['transmission_type'] == 'Other':
        key = (row['model_year'], row['brand'], row['model'])
        return transmission_type_map.get(key, row['transmission_type'])
    return row['transmission_type']

# Apply the imputation function
df_update['transmission_type'] = df_update.apply(lambda row: impute_transmission_type(row, transmission_type_map), axis=1)

In [791]:
## Check for Missing Transmission Types
mask = df_update['transmission_type'] == 'Other'

df_update.loc[mask,['model_year', 'brand', 'model', 'transmission', 'transmission_type']]

,model_year,brand,model,transmission,transmission_type


In [792]:
## Dictionary mapping (brand, model) to transmission type
# map_transmission_type = {
#     (2016, 'Acura', 'ILX 2.4L'): 'Automatic',
#     (2022, 'Acura', 'TLX w/A-Spec Package'): 'Automatic',
#     (2023, 'Acura', 'MDX w/Technology Package'): 'Automatic',
#     (2022, 'Acura', 'MDX w/Technology Package'): 'Automatic',
#     (1974, 'Ford', 'Bronco'): 'Automatic',
#     (2018, 'Porsche', '911 Carrera 4S'): 'Automatic',
#     (2019, 'Toyota', 'Tacoma TRD Pro'): 'Automatic',
#     (2021, 'Lamborghini', 'Aventador SVJ Base'): 'Automatic',
#     (2023, 'Rolls-Royce', 'Phantom'): 'Automatic',
#     (2021, 'Acura', 'RDX PMC Edition'): 'Automatic',
# }

## Update transmission_type dictionary
# transmission_type_map.update(map_transmission_type)

# Save the new transmission type mapping
# with open(os.path.join(assets_dir, 'transmission_type_map.pkl'), 'wb') as f:
#     pickle.dump(transmission_type_map, f)


In [793]:


# def assign_transmission_type(row):
#     """
#     Assigns the transmission type based on the brand and model.

#     Args:
#         row (pd.Series): A row of the DataFrame.

#     Returns:
#         str: The assigned transmission type.
#     """
#     if row['transmission_type'] == 'Other':
#         key = (row['model_year'], row['brand'], row['model'])
#         return transmission_type_map.get(key, row['transmission_type'])
#     else:
#         return row['transmission_type']

# ## Apply the mapping to the DataFrame
# df_update['transmission_type'] = df_update.apply(assign_transmission_type, axis=1)



In [794]:
df_update['transmission_type'].value_counts() 

transmission_type
Automatic    3568
Manual        374
CVT            67
Name: count, dtype: int64

✅ Transmission Type Standardization Complete

- All transmission type values have been reviewed and corrected.  
- The column is now ready for analysis and modeling.

---

##### Fix transmission speeds

In [795]:
## Check if same models already have transmission speeds
try:
    with open(os.path.join(assets_dir, 'transmission_speeds_map.pkl'), 'rb') as f:
        transmission_speeds_map = pickle.load(f)
except FileNotFoundError:
    print("Dictonary not available.\nCreating transmission speeds map...")
    transmission_speeds_map = (
        df_update[df_update['transmission_speeds'].notna()]
        .groupby(['model_year', 'brand', 'model'])['transmission_speeds']
        .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
        .to_dict()
    )


In [796]:
## Impute transmission_speeds
def impute_transmission_speeds(row, transmission_speeds_map):
    """
    Impute the transmission speeds based on the brand and model.
    
    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        int or None: The imputed transmission speeds.
    """
    if pd.isna(row['transmission_speeds']):
        key = (row['model_year'], row['brand'], row['model'])
        return transmission_speeds_map.get(key, row['transmission_speeds'])
    else:
        return row['transmission_speeds']
    
# Apply the imputation function
df_update['transmission_speeds'] = df_update.apply(lambda row: impute_transmission_speeds(row, transmission_speeds_map), axis=1)

In [797]:
# ## Dictionary mapping (brand, model) to transmission speeds
# """
# To save space, just display the format of the dictionary.
# """
# map_transmission_speeds = {
#     (2018, 'Mercedes-Benz', 'E-Class E 300 4MATIC'): 9,
#     (2022, 'Audi', 'S4 3.0T Premium Plus'): 8,
#     (2022, 'Porsche', 'Taycan'): 1,
#     (2020, 'Ford', 'F-150 Raptor'): 10,
# }

## Update dictionary with additional vehicles
# transmission_speeds_map.update(map_transmission_speeds)

# # Save the new transmission type mapping
# with open(os.path.join(assets_dir, 'transmission_speeds_map.pkl'), 'wb') as f:
#     pickle.dump(transmission_speeds_map, f)

In [799]:
df_update['transmission_speeds'].isna().sum()

0

✅ All transmission speeds for colossal list of car models have been assigned and are ready to roll. 


---



In [800]:
df_update.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,transmission_type,transmission_speeds
0,Ford,Utility Police Interceptor Base,2013,51000.0,E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,10300.0,Automatic,6.0
1,Hyundai,Palisade SEL,2021,34742.0,Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,38005.0,Automatic,8.0
2,Lexus,RX 350,2022,22372.0,Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,54598.0,Automatic,8.0
3,INFINITI,Q50 Hybrid Sport,2015,88900.0,Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,15500.0,Automatic,7.0
4,Audi,Q3 45 S line Premium Plus,2021,9835.0,Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,34999.0,Automatic,8.0


In [801]:
df_update['fuel_type'].unique()

array(['E85 Flex Fuel', 'Gasoline', 'Hybrid', 'Electric', 'Diesel',
       'Plug-In Hybrid', 'Hydrogen'], dtype=object)

### Engine Data
Engine data has rich info. it can be extract to various features. e.g. hourse power (HP), Liter, Cylinder

In [802]:
def engine_extract(row):
    """
    Extracts engine features from the 'engine' column.
    
    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        dict: A dictionary with extracted engine features.
    """
    ## Handle Electric Vehicles
    if row['fuel_type'] in ['Electric', 'Hydrogen']:
        match = re.search(r'(\d+\.?\d*)\s*hp', row['engine'], re.IGNORECASE)
        return {
            'hp': float(match.group(1)) if match else None,
            'liters': 0.0,
            'cylinders': 0.0
        }

    match = re.search(
        r'(?:([\d.]+)\s*hp\s+)?([\d.]+)?\s*(?:l|liter)?\s*(?:v|i|flat|straight|rotary)?\s*(\d+)?(?:\s*cylinder)?',
        row['engine'],
        re.IGNORECASE
    )

    if match:
        hp = float(match.group(1)) if match.group(1) else None
        liters = float(match.group(2)) if match.group(2) else None
        cylinders = float(match.group(3)) if match.group(3) else None
        
        ## Double check if the parse value is correct
        if liters is not None and not (0<= liters <= 10):
            liters = None
        
        if cylinders is not None and not (0 <= cylinders <= 16):
            cylinders = None

        return {'hp': hp, 'liters': liters, 'cylinders': cylinders}

    return {'hp': None, 'liters': None, 'cylinders': None}

In [803]:
## Apply engine extraction to the DataFrame
df_update[['hp', 'liters', 'cylinders']] = df_update.apply(engine_extract, axis=1, result_type='expand')

In [804]:
## Check for missing values in the new columns
df_update[['hp', 'liters', 'cylinders']].isna().sum()

hp           810
liters        59
cylinders    275
dtype: int64

#### 🛠️ Engine Data: Imputation & Standardization

Engine specs (`hp`, `liters`, `cylinders`) can be missing or inconsistent across similar vehicles.
To ensure data completeness and accuracy, I’ll impute missing values using a prioritized approach:

1. **Build Engine Spec Mapping**  
   - Create a dictionary mapping of available engine specs for each unique combo of model, year, brand, liters, and cylinders.

2. **Identify Models With All Engine Specs Missing**  
   - Find vehicles with all three specs (`hp`, `liters`, `cylinders`) missing.
   - Impute these by looking up the records (by brand/model/year/engine config).

3. **Impute Partially Missing Specs**  
   - For vehicles with one or two missing engine specs:
     - Impute **liters** first, then **cylinders**, then **hp**, following the order of most complete -> least complete.
     - For each missing value, search for matching vehicles (brand, model, liters, cylinders) using the closest current/prior model year.
     - If multiple candidates exist, use the most frequent value (mode). If still tied, use the highest value.

4. **Update the Engine Map After Each Imputation**  
   - If a row is completed with all specs, add it to the engine spec mapping for future reference.

**Example:**
> If a 2019 Ford F-150 XLT has 3.5L, 6 cylinders, but missing `hp`, look up the most similar prior records.  
> If find `325` and `375` as options and both are equally frequent, impute with `375`.

##### 🛠️ Create Engine Spec Maps

To efficiently impute missing engine specs (horsepower, liters, cylinders), I build a mapping structure that summarizes all known engine configurations for each (model_year, brand, model) combo in the dataset.

The resulting map is a nested dictionary that looks like this:

```
{
  (2019, 'Ford', 'F-150 XLT'): {
      (325.0, 2.7, 6.0): 1,
      (335.0, 2.7, 6.0): 2,
      (375.0, 3.5, 6.0): 3,
      (395.0, 5.0, 8.0): 4
  },
  (2021, 'BMW', 'X5 xDrive40i'): {
      (335.0, 3.0, 6.0): 5
  },
  ...
}
```

In [805]:
## Check if same models already have transmission speeds
try:
    with open(os.path.join(assets_dir, 'engine_specs.pkl'), 'rb') as f:
        engine_specs = pickle.load(f)
except FileNotFoundError:
    print("Dictonary not available.\nCreating engine specs map...")
    
    engine_specs = defaultdict(Counter)

    for row in df_update[['model_year', 'brand', 'model', 'hp', 'liters', 'cylinders']].itertuples(index=False):
        model_year, brand, model, hp, liters, cylinders = row
        if pd.notna(hp) and pd.notna(liters) and pd.notna(cylinders):
            key = (model_year, brand, model)
            spec = (hp, liters, cylinders)
            engine_specs[key][spec] += 1



In [ ]:
## Function to impute all 3 null engine specs
def impute_full_null_specs(row, engine_specs):
    """
    Impute missing engine specs (hp, liters, cylinders) when ALL THREE are missing.
    Looks for the most common engine spec for the (model_year, brand, model).
    If there’s a tie, picks the biggest (hp > liters > cylinders).
    If not found for this year, checks +/- 1, 2, 3 years out.
    """
    key = (row['model_year'], row['brand'], row['model'])
    # Grab all engine spec combos/counts for this exact year-brand-model
    spec_cnt = engine_specs.get(key, {})

    # If nothing for this year-brand-model, search ±3 years
    if not spec_cnt:
        for delta in range(1, 4):
            for shift in [row['model_year']-delta, row['model_year']+delta]:
                alt_key = (shift, row['brand'], row['model'])
                spec_cnt = engine_specs.get(alt_key, {})
                if spec_cnt:
                    break  # Stop looking further if you found any
            if spec_cnt:
                break

    if spec_cnt:  # Got some data, time to pick the winner
        max_cnt = max(spec_cnt.values())  # Highest count (mode)
        # Grab all engine specs with that highest count
        mode_specs = [spec for spec, cnt in spec_cnt.items() if cnt == max_cnt]
        # If tie, pick biggest hp, then liters, then cylinders
        best_spec = max(mode_specs, key=lambda x: (x[0], x[1], x[2]))
        return {'hp': best_spec[0], 'liters': best_spec[1], 'cylinders': best_spec[2]}

    # Fallback: couldn't impute, return all None
    return {'hp': None, 'liters': None, 'cylinders': None}


## Function to impute partial null engine specs
def impute_partial_null_specs(row, engine_specs):
    """
    Impute engine specs when one or two are missing.
    Looks for the most common spec matching the other columns.
    Prioritizes same MY first, then checks ±3 years.
    Mode wins; if tie, picks biggest value.
    """
    key = (row['model_year'], row['brand'], row['model'])
    spec_cnt = engine_specs.get(key, {})

    # Put current values into a list so we can index it
    target = [row['hp'], row['liters'], row['cylinders']]
    nulls = [idx for idx, v in enumerate(target) if pd.isna(v)]  # Indices of missing values
    filled = target.copy()  # Start with what we have

    # If nothing for this year, search ±3 years
    if not spec_cnt:
        for delta in range(1, 4):
            for shift in [row['model_year']-delta, row['model_year']+delta]:
                alt_key = (shift, row['brand'], row['model'])
                spec_cnt = engine_specs.get(alt_key, {})
                if spec_cnt:
                    break
            if spec_cnt:
                break

    # If we found possible specs and not all three are missing
    if spec_cnt and len(nulls) < 3:
        for idx in nulls:  # For each missing spec
            # Build up all (value, count) that match the KNOWN specs
            specs = []
            for spec, cnt in spec_cnt.items():
                match = True
                for j in range(3):
                    if j != idx and pd.notna(target[j]) and spec[j] != target[j]:
                        match = False
                        break
                if match:
                    specs.append((spec[idx], cnt))
            if specs:
                # mode or max if tie
                max_cnt = max(cnt for val, cnt in specs)
                # Grab all values with highest count, pick the biggest one
                filled[idx] = max([val for val, cnt in specs if cnt == max_cnt])

        return {'hp': filled[0], 'liters': filled[1], 'cylinders': filled[2]}

    # Return the original if nothing is found
    return {'hp': row['hp'], 'liters': row['liters'], 'cylinders': row['cylinders']}

In [807]:
def impute_engine_specs(row, engine_specs):
    if row[['hp', 'liters', 'cylinders']].isna().sum() == 3:
        return impute_full_null_specs(row, engine_specs)
    elif row[['hp', 'liters', 'cylinders']].isna().sum() in [1,2]:
        return impute_partial_null_specs(row, engine_specs)
    else:
        return {'hp': row['hp'], 'liters': row['liters'], 'cylinders': row['cylinders']}
    
df_update[['hp', 'liters', 'cylinders']] = df_update.apply(lambda row: impute_engine_specs(row, engine_specs), axis=1, result_type='expand')

##### 🛠️ Manual Imputation Needed

After automated imputation, some engine specs are still missing for certain models.  
For these remaining gaps, manual imputation is required. Please review the unresolved records and update the missing values as needed.

> ❕ Note:  
> For the complete set of manual mappings and updates, see `manual_map_engine_specs.py`.

In [ ]:
# ## Create a dictionary mapping (model_year, brand, model) to engine specs for missing engine data models
# map_engine_specs = {
#     (2022, 'RAM', '1500 Big Horn'): Counter({(395.0, 5.7, 8.0): 1}),
#     (2021, 'Ford', 'F-150 XLT'): Counter({(400.0, 3.5, 6.0): 1}),
#     (2020, 'Honda', 'Civic Sport'): Counter({(158.0, 2.0, 4.0): 1}),
# }

# ## update engine_specs with the new data
# for key, counter in map_engine_specs.items():
#     engine_specs[key].update(counter)

# # save engine_specs dictionary
# with open(os.path.join(assets_dir, 'engine_specs.pkl'), 'wb') as f:
#     pickle.dump(engine_specs, f)

## 🚗 Accident History Imputation

There are three types of accident values in the dataset:
- **'None reported'** - will be labeled as 0.
- **'At least 1 accident or damage reported'** - will be labeled as 1.
- **NaN (missing value)** - will be considered as 'had accident' and be labeled as 1.

For modeling purposes, *missing values (`NaN`)* will be **imputed as 'At least 1 accident or damage reported'**.  

This is because accident history is one of the most critical factors in used car evaluation. If a seller (or previous owner) leaves this field blank, it raises a major red flag—potentially indicating an attempt to hide accident history.

Treating missing data as “damage reported” makes the model more aggressive and risk-averse, which ultimately protects both buyers and sellers from misrepresentation or hidden issues.  

Leaving this field blank could easily mislead the model and users. If you’re not willing to admit “none,” we assume the worst.

In [809]:
df_update['accident'].value_counts(dropna=False)

accident
None reported                             2910
At least 1 accident or damage reported     986
NaN                                        113
Name: count, dtype: int64

In [810]:
def impute_values(series, value):
    """
    Impute missing values in a series with a specified value.
    
    Args:
        series (pd.Series): The series to impute.
        value: The value to use for imputation.
        
    Returns:
        pd.Series: The series with missing values filled.
    """
    return series.fillna(value)

## Impute missing values in the 'accident' column
df_update['accident'] = impute_values(df_update['accident'], 'At least 1 accident or damage reported')

In [811]:
## Convert 'accident' column to categorical type
df_update['accident_reported'] = df_update['accident'].map({
    'None reported': '0',
    'At least 1 accident or damage reported': '1'
})

## 🚗 Clean_title Imputation

Just like with the `Accident` column, there are only two possible values for clean title in this dataset:
- **'Yes'** - will be labeled as 1
- **NaN** (missing) - will be labeled as 0

For modeling, every *missing value (`NaN`)* will be **imputed as 'No'**.  

Why? Because clean title status is one of the most critical damn factors for evaluating a used car. If someone leaves this blank, they’re probably trying to hide something, and that is a massive red flag. 

Imputing missing values as “No” (not a clean title) forces the model to be more aggressive and risk-averse—just the way it should be when there’s potential for someone to cover up damage or dirty history. This approach protects both the buyer and the seller from shady surprises.

Bottom line: If you can't claim that it’s a clean title, we’re gonna assume the worst.  

In [812]:
df_update['clean_title'].value_counts(dropna=False)

clean_title
Yes    3413
NaN     596
Name: count, dtype: int64

In [813]:
df_update['clean_title'] = df_update['clean_title'].fillna('No')

df_update['clean_title'] = df_update['clean_title'].map({
        'Yes': 1,
        'No': 0
        })


In [814]:
df_update['clean_title'].value_counts(dropna=False)

clean_title
1    3413
0     596
Name: count, dtype: int64

In [815]:
df_update.isna().sum()

brand                  0
model                  0
model_year             0
milage                 0
fuel_type              0
engine                 0
transmission           0
ext_col                0
int_col                0
accident               0
clean_title            0
price                  0
transmission_type      0
transmission_speeds    0
hp                     0
liters                 0
cylinders              0
accident_reported      0
dtype: int64

# ✅ Data Preprocessing Complete

All major data cleaning and preprocessing steps are now finished.  
Missing values have been imputed, features have been engineered, and the dataset is ready for analysis.

---

**Next Steps:**

1. **Exploratory Data Analysis (EDA):**  
   - Dive into the data to understand distributions, spot outliers, visualize relationships, and identify potential patterns or anomalies that could impact modeling.

2. **Machine Learning:**  
   - Build, train, and evaluate predictive models using the cleaned dataset.  
   - Experiment with different algorithms, tune hyperparameters, and validate model performance.

